In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_boston
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# データの読み込み（例としてBostonデータセットを使用）
data = load_boston()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# データの分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoostモデルのインスタンスを作成
xg_reg = xgb.XGBRegressor(objective='reg:squarederror', n_jobs=-1)  # n_jobs=-1で全てのスレッドを使用

# ハイパーパラメータのグリッド
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# GridSearchCVの設定
grid_search = GridSearchCV(
    estimator=xg_reg,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',  # RMSEをスコアリングメトリックとして使用
    cv=5,  # 5-foldクロスバリデーション
    n_jobs=-1,  # 全てのスレッドを使用
    verbose=1  # 学習状況の表示
)

# ハイパーパラメータチューニング
grid_search.fit(X_train, y_train)

# 最適なハイパーパラメータ
print("Best Parameters:\n", grid_search.best_params_)

# 最適なモデルで予測
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# RMSEの計算
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", rmse)


In [ ]:
import lightgbm as lgb
from sklearn.datasets import load_boston
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error
import numpy as np

# データの読み込み（ここではボストンのサンプルデータを使用していますが、実際のデータを読み込むことができます）
X, y = load_boston(return_X_y=True)

# LightGBMのモデルを定義
lgbm = lgb.LGBMRegressor()

# ハイパーパラメータのグリッドを定義
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [-1, 10, 20],
    'learning_rate': [0.01, 0.1, 0.2],
    'num_leaves': [31, 63, 127],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# グリッドサーチの設定
grid_search = GridSearchCV(estimator=lgbm,
                           param_grid=param_grid,
                           scoring='neg_root_mean_squared_error',  # RMSEで評価
                           cv=5,  # クロスバリデーションの分割数
                           n_jobs=-1)  # 全てのCPUコアを使用

# グリッドサーチの実行
grid_search.fit(X, y)

# ベストパラメータとベストスコアの表示
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best RMSE: {-grid_search.best_score_}')  # GridSearchCVはスコアが負で返されるため、マイナスをつけて表示

# ベストモデルでの予測
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f'Final RMSE on the training set: {rmse}')
